# Лекција 09 - Образац дизајна метакогниције


## Подешавање

Овај нотебук демонстрира образац дизајна метакогниције користећи Microsoft Agent Framework.

**Предуслови:**
- Azure OpenAI deployment конфигурисан помоћу променљивих окружења
- Azure CLI аутентификован (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Шта је метакогниција?

Метакогниција је **размишљање о размишљању**. У контексту AI агената, то значи изградњу агената који могу:

- **Саморазмишљати** о својим излазима и процесу расуђивања
- **Препознавати грешке** и опорављати се на прикладан начин уместо да ћутке пропадну
- **Проценити** да ли су њихови одговори потпуни и корисни
- **Прилагодити** своју стратегију када почетни приступ не функционише (нпр. прелазак на резервни систем)

Метакогнитивни агент не само да одговара на питања — он прати своје перформансе и прилагођава се у ходу.


## Примарни и резервни алати

Уобичајен образац метакогниције је **резервна стратегија**. Агент прво покушава примарни алат; ако он не успе (нпр. грешка 404), агент препознаје неуспех и транспарентно прелази на резервни алат.

Ово одсликава системе из стварног света где примарне услуге могу бити недоступне и агенти морају сами дијагностиковати проблем пре него што изаберу алтернативни пут.

Испод дефинишемо два алата за претрагу летова:
- **Примарни** — покрива Париз, Токио и Барселону
- **Резервни** — покрива Берлин, Сиднеј и Њујорк


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Саморефлектујући агент са опоравком од грешака

Агент испод је упућен да прво покуша да користи примарни летни систем, препозна неуспехе и транспарентно пређе на резервни систем. Након сваког одговора, он кратко самооцени да ли је у потпуности одговорио на питање корисника.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Шаблон самопроцене

Још један аспект метакогниције је **самопроцена**: одвојени агент (или исти агент у другом пролазу) прегледа одговор ради потпуности, тачности и корисности.

Испод стварамо агента `ResponseEvaluator` који оцењује одговоре агента за путовања по три димензије.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Резиме

У овој лекцији сте научили како да изградите **метакогнитивне агенте** користећи Microsoft Agent Framework:

- **Саморефлексија**: Агенти који надгледају сопствено расуђивање и транспарентно комуницирају шта се догодило.
- **Опоравак од грешака уз резервне опције**: Образац примарног + резервног алата у којем агент детектује неуспехе (нпр. 404 грешке) и аутоматски покушава алтернативни извор.
- **Самоевалуација**: Посебан агент за евалуацију који оцењује одговоре по комплетности, тачности и корисности.

Ови обрасци чине агенте робуснијим, транспарентнијим и поузданијим — критичне карактеристике за продукционо распоређивање.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Одрицање од одговорности:
Овај документ је преведен уз помоћ сервиса за превођење заснованог на вештачкој интелигенцији Co‑op Translator (https://github.com/Azure/co-op-translator). Иако настојимо да будемо тачни, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати меродавним извором. За критичне информације препоручује се професионалан превод који обави стручни преводилац. Не сносимо одговорност за било какве неспоразуме или погрешна тумачења која произилазе из употребе овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
